# Gold Layer Quality Checks

This notebook validates the complete Gold layer, including model availability, row counts, grain uniqueness, key completeness, referential integrity, date-key coverage, and numerical business rules.

Some fact records may reference orders quarantined during Silver processing. These are reported separately rather than silently removed.

In [0]:
from pyspark.sql import functions as F

## 1. Define Gold storage paths

In [0]:
GOLD_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist"
)

GOLD_PATHS = {
    "dim_customers": f"{GOLD_BASE_PATH}/dim_customers",
    "dim_sellers": f"{GOLD_BASE_PATH}/dim_sellers",
    "dim_products": f"{GOLD_BASE_PATH}/dim_products",
    "dim_dates": f"{GOLD_BASE_PATH}/dim_dates",
    "fact_orders": f"{GOLD_BASE_PATH}/fact_orders",
    "fact_order_items": f"{GOLD_BASE_PATH}/fact_order_items",
    "fact_payments": f"{GOLD_BASE_PATH}/fact_payments",
    "fact_reviews": f"{GOLD_BASE_PATH}/fact_reviews",
    "sales_daily": f"{GOLD_BASE_PATH}/sales_daily",
    "sales_by_state": f"{GOLD_BASE_PATH}/sales_by_state",
    "product_performance": f"{GOLD_BASE_PATH}/product_performance",
    "seller_performance": f"{GOLD_BASE_PATH}/seller_performance",
    "delivery_performance": f"{GOLD_BASE_PATH}/delivery_performance",
}

## 2. Read all Gold models

In [0]:
gold_dfs = {}

for model_name, model_path in GOLD_PATHS.items():
    try:
        model_df = (
            spark.read
            .format("delta")
            .load(model_path)
        )

        gold_dfs[model_name] = model_df

        print(
            f"{model_name}: {model_df.count():,} rows"
        )

    except Exception as exc:
        raise ValueError(
            f"Failed to read Gold model '{model_name}' "
            f"from {model_path}"
        ) from exc

In [0]:
dim_customers_df = gold_dfs["dim_customers"]
dim_sellers_df = gold_dfs["dim_sellers"]
dim_products_df = gold_dfs["dim_products"]
dim_dates_df = gold_dfs["dim_dates"]

fact_orders_df = gold_dfs["fact_orders"]
fact_order_items_df = gold_dfs["fact_order_items"]
fact_payments_df = gold_dfs["fact_payments"]
fact_reviews_df = gold_dfs["fact_reviews"]

## 3. Validate non-empty Gold models

In [0]:
for model_name, model_df in gold_dfs.items():
    row_count = model_df.count()

    if row_count == 0:
        raise ValueError(
            f"Gold model '{model_name}' is empty."
        )

    print(
        f"{model_name}: non-empty validation passed."
    )

## 4. Validate dimension and fact grains

In [0]:
grain_checks = [
    ("dim_customers", dim_customers_df, ["customer_id"]),
    ("dim_sellers", dim_sellers_df, ["seller_id"]),
    ("dim_products", dim_products_df, ["product_id"]),
    ("dim_dates", dim_dates_df, ["date_key"]),
    ("fact_orders", fact_orders_df, ["order_id"]),
    (
        "fact_order_items",
        fact_order_items_df,
        ["order_id", "order_item_id"],
    ),
    (
        "fact_payments",
        fact_payments_df,
        ["order_id", "payment_sequential"],
    ),
    (
        "fact_reviews",
        fact_reviews_df,
        ["review_id", "order_id"],
    ),
]

for model_name, model_df, grain_columns in grain_checks:
    duplicate_count = (
        model_df
        .groupBy(*grain_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_key_condition = None

    for column_name in grain_columns:
        condition = F.col(column_name).isNull()

        null_key_condition = (
            condition
            if null_key_condition is None
            else null_key_condition | condition
        )

    null_key_count = (
        model_df
        .filter(null_key_condition)
        .count()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{model_name} contains "
            f"{duplicate_count:,} duplicate grain combinations."
        )

    if null_key_count > 0:
        raise ValueError(
            f"{model_name} contains "
            f"{null_key_count:,} rows with null grain keys."
        )

    print(
        f"{model_name}: grain validation passed."
    )

## 5. Validate dimension references

In [0]:
dimension_reference_checks = [
    (
        "fact_orders.customer_id",
        fact_orders_df,
        "customer_id",
        dim_customers_df,
        "customer_id",
    ),
    (
        "fact_order_items.product_id",
        fact_order_items_df,
        "product_id",
        dim_products_df,
        "product_id",
    ),
    (
        "fact_order_items.seller_id",
        fact_order_items_df,
        "seller_id",
        dim_sellers_df,
        "seller_id",
    ),
]

for (
    relationship_name,
    source_df,
    source_key,
    target_df,
    target_key,
) in dimension_reference_checks:

    missing_reference_count = (
        source_df
        .select(source_key)
        .filter(F.col(source_key).isNotNull())
        .distinct()
        .join(
            target_df
            .select(F.col(target_key).alias(source_key))
            .distinct(),
            on=source_key,
            how="left_anti",
        )
        .count()
    )

    if missing_reference_count > 0:
        raise ValueError(
            f"{relationship_name} has "
            f"{missing_reference_count:,} missing dimension references."
        )

    print(
        f"{relationship_name}: referential integrity passed."
    )

## 6. Profile fact records linked to quarantined orders

In [0]:
valid_order_ids_df = (
    fact_orders_df
    .select("order_id")
    .distinct()
)

fact_order_reference_profiles = [
    (
        "fact_order_items",
        fact_order_items_df,
    ),
    (
        "fact_payments",
        fact_payments_df,
    ),
    (
        "fact_reviews",
        fact_reviews_df,
    ),
]

orphan_order_summary = []

for model_name, model_df in fact_order_reference_profiles:
    orphan_df = (
        model_df
        .join(
            valid_order_ids_df,
            on="order_id",
            how="left_anti",
        )
    )

    orphan_row_count = orphan_df.count()

    orphan_order_count = (
        orphan_df
        .select("order_id")
        .distinct()
        .count()
    )

    orphan_order_summary.append(
        (
            model_name,
            orphan_row_count,
            orphan_order_count,
        )
    )

    print(
        f"{model_name}: {orphan_row_count:,} rows across "
        f"{orphan_order_count:,} orders do not match fact_orders."
    )

orphan_order_summary_df = spark.createDataFrame(
    orphan_order_summary,
    [
        "model_name",
        "orphan_row_count",
        "orphan_order_count",
    ],
)

display(orphan_order_summary_df)

## 7. Validate date-key references

In [0]:
date_keys_df = (
    dim_dates_df
    .select("date_key")
    .distinct()
)

date_reference_checks = [
    (
        "fact_orders.purchase_date_key",
        fact_orders_df,
        "purchase_date_key",
    ),
    (
        "fact_orders.approved_date_key",
        fact_orders_df,
        "approved_date_key",
    ),
    (
        "fact_orders.delivered_carrier_date_key",
        fact_orders_df,
        "delivered_carrier_date_key",
    ),
    (
        "fact_orders.delivered_customer_date_key",
        fact_orders_df,
        "delivered_customer_date_key",
    ),
    (
        "fact_orders.estimated_delivery_date_key",
        fact_orders_df,
        "estimated_delivery_date_key",
    ),
    (
        "fact_order_items.shipping_limit_date_key",
        fact_order_items_df,
        "shipping_limit_date_key",
    ),
    (
        "fact_reviews.review_creation_date_key",
        fact_reviews_df,
        "review_creation_date_key",
    ),
    (
        "fact_reviews.review_answer_date_key",
        fact_reviews_df,
        "review_answer_date_key",
    ),
]

for relationship_name, source_df, date_key_column in date_reference_checks:
    missing_date_count = (
        source_df
        .select(
            F.col(date_key_column).alias("date_key")
        )
        .filter(F.col("date_key").isNotNull())
        .distinct()
        .join(
            date_keys_df,
            on="date_key",
            how="left_anti",
        )
        .count()
    )

    if missing_date_count > 0:
        raise ValueError(
            f"{relationship_name} has "
            f"{missing_date_count:,} date keys missing from dim_dates."
        )

    print(
        f"{relationship_name}: date-key validation passed."
    )

## 8. Validate numerical business rules

In [0]:
invalid_order_item_value_count = (
    fact_order_items_df
    .filter(
        F.col("price").isNull()
        | (F.col("price") < 0)
        | F.col("freight_value").isNull()
        | (F.col("freight_value") < 0)
        | F.col("item_total_value").isNull()
        | (F.col("item_total_value") < 0)
    )
    .count()
)

invalid_payment_value_count = (
    fact_payments_df
    .filter(
        F.col("payment_value").isNull()
        | (F.col("payment_value") < 0)
    )
    .count()
)

invalid_review_score_count = (
    fact_reviews_df
    .filter(
        F.col("review_score").isNull()
        | ~F.col("review_score").between(1, 5)
    )
    .count()
)

if invalid_order_item_value_count > 0:
    raise ValueError(
        "fact_order_items contains "
        f"{invalid_order_item_value_count:,} invalid monetary rows."
    )

if invalid_payment_value_count > 0:
    raise ValueError(
        "fact_payments contains "
        f"{invalid_payment_value_count:,} invalid payment rows."
    )

if invalid_review_score_count > 0:
    raise ValueError(
        "fact_reviews contains "
        f"{invalid_review_score_count:,} invalid review-score rows."
    )

print("Order-item monetary validation passed.")
print("Payment-value validation passed.")
print("Review-score validation passed.")

## 9. Validate aggregate grains

In [0]:
aggregate_grain_checks = [
    (
        "sales_daily",
        gold_dfs["sales_daily"],
        ["date_key"],
    ),
    (
        "sales_by_state",
        gold_dfs["sales_by_state"],
        ["customer_state"],
    ),
    (
        "product_performance",
        gold_dfs["product_performance"],
        ["product_id"],
    ),
    (
        "seller_performance",
        gold_dfs["seller_performance"],
        ["seller_id"],
    ),
    (
        "delivery_performance",
        gold_dfs["delivery_performance"],
        ["purchase_date_key"],
    ),
]

for model_name, model_df, grain_columns in aggregate_grain_checks:
    duplicate_count = (
        model_df
        .groupBy(*grain_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{model_name} contains "
            f"{duplicate_count:,} duplicate grain combinations."
        )

    print(
        f"{model_name}: aggregate grain validation passed."
    )

## 10. Create Gold quality summary

In [0]:
quality_summary_rows = []

for model_name, model_df in gold_dfs.items():
    quality_summary_rows.append(
        (
            model_name,
            model_df.count(),
            len(model_df.columns),
            "PASSED",
        )
    )

gold_quality_summary_df = (
    spark.createDataFrame(
        quality_summary_rows,
        [
            "model_name",
            "row_count",
            "column_count",
            "quality_status",
        ],
    )
    .withColumn(
        "checked_at",
        F.current_timestamp(),
    )
)

display(
    gold_quality_summary_df
    .orderBy("model_name")
)

## 11. Final validation result

In [0]:
print("All critical Gold layer quality checks passed.")
print(
    "Order references affected by Silver order quarantine "
    "were profiled separately."
)